# When does simple win? Three tabular datasets at increasing difficulty

This notebook runs `stepzero` on three well-known classification datasets and lets the
headroom signal tell you what to do next.

| Dataset | Expected difficulty | Why |
|---------|-------------------|-----|
| Wine | Easy | Linearly separable classes |
| Titanic | Medium | Missing values, non-linear patterns |
| California Housing | Hard | Complex non-linear relationships |


In [ ]:
import stepzero as sz
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Dataset 1 — Wine (easy)

Three wine cultivars separated by chemical measurements. Logistic regression should nail this.

In [ ]:
from sklearn.datasets import load_wine
X, y = load_wine(return_X_y=True, as_frame=True)
print(f"{len(X)} samples, {X.shape[1]} features, {len(set(y))} classes")
X.head(3)


In [ ]:
result = sz.classification(X, y)
print(result)
print()
print(result.headroom)


In [ ]:
result.feature_importance.head(8).plot(kind="barh", title="Wine: top features")


## Dataset 2 — Titanic (medium)

The Kaggle classic. Simple models get ~80%; feature engineering and ensembles push toward 83–85%.

In [ ]:
import seaborn as sns
titanic = sns.load_dataset("titanic")

# Basic feature prep — keep the columns a simple model can use
features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
target = "survived"

df = titanic[features + [target]].dropna(subset=[target])
X = df[features]
y = df[target]
print(f"{len(X)} samples after dropping missing target")
X.head(3)


In [ ]:
result = sz.classification(X, y)
print(result)
print()
print(result.headroom)


In [ ]:
if result.feature_importance is not None:
    result.feature_importance.head(10).plot(kind="barh", title="Titanic: top features")


### What stepzero found
The headroom signal should be **Medium** — simple models plateau around 80% accuracy on Titanic.
Features like `title` (extracted from name), `family_size`, and `deck` are the usual gains.


## Dataset 3 — California Housing (hard)

Predict house prices from 8 features. Non-linear geography effects (latitude/longitude) mean
linear models leave significant RMSE on the table.

In [ ]:
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target
print(f"{len(X)} samples, {X.shape[1]} features")
print(f"Target range: ${y.min()*100_000:.0f} – ${y.max()*100_000:.0f}")
X.head(3)


In [ ]:
result = sz.regression(X, y)
print(result)
print()
print(result.headroom)


In [ ]:
result.feature_importance.plot(kind="barh", title="California Housing: feature importance")


In [ ]:
# Visualise residuals of the best model
y_pred = result.best_model.predict(X)
residuals = y - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_pred, residuals, alpha=0.1, s=5)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("predicted")
axes[0].set_ylabel("residual")
axes[0].set_title(f"Residuals — {result.best_model_name}")

axes[1].hist(residuals, bins=60, edgecolor="none")
axes[1].set_title("Residual distribution")


### Summary

| Dataset | Best model | Score | Headroom |
|---------|-----------|-------|----------|
| Wine | — | — | — |
| Titanic | — | — | — |
| California Housing | — | — | — |

Fill in from the outputs above. The progression from Low → Medium → High headroom shows
exactly when it's worth reaching for gradient boosting.
